# 3. Dataset Processing: The Latent Oddity Approach

We utilize the two-stage VAE pipeline trained previously (Continuous VAE + Post-Hoc Riemannian Quantizer) to process our raw text dataset.

The goal of this phase is to encode the raw training data into discrete latent tokens. Unlike standard Euclidean approaches (like standard VQ-VAEs), our **Latent Oddity Quantizer** assigns tokens based on the stochastic Riemannian geometry of the decoder. This ensures that the generated discrete dataset accurately reflects the curved data manifold, avoiding "dead" or untrained regions of the latent space.

## 3.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/content
Cloning into 'DLAI'...
remote: Enumerating objects: 637, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 637 (delta 135), reused 0 (delta 0), pack-reused 446 (from 2)
Receiving objects: 100% (637/637), 280.11 KiB | 2.15 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/DLAI
Using Python 3.12.13 environment at: /usr
Resolved 87 packages in 743ms
Prepared 3 packages in 1.47s
Installed 3 packages in 7ms
 + bitsandbytes==0.49.2
 + dlai-metamath==0.1.0 (from file:///content/DLAI)
 + trl==0.29.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content


### 1. Load Tokenizer and Riemannian Models

Before processing the text, we need to load the backbone models:
1. **Continuous VAE**: Provides the continuous, smooth encoding of the text sequences.
2. **Latent Oddity Quantizer**: Maps the continuous representations into discrete tokens using the pre-computed Riemannian metric (based on the Jacobian and Variance of the decoder).

In [ ]:
import torch
import json

from src.utils import get_llm_tokenizer, MAX_SEQ_LEN, VQ_CODEBOOK_SIZE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define file paths for the saved models
PATH_CONTINUOUS_VAE = "/content/drive/MyDrive/DLAI/experiments/oddity/vae_continuous_uv.pth"
PATH_ODDITY_QUANTIZER = "/content/drive/MyDrive/DLAI/experiments/oddity/oddity_quantizer_posthoc_uv.pth"

tokenizer = get_llm_tokenizer()
vocab_size = len(tokenizer)

# 1. Load the Continuous VAE (Backbone)
# This model provides the smooth, continuous manifold mapped from the input text.
from src.model.vae_oddity import ContinuousVAE
vae_model = ContinuousVAE(
    vocab_size=vocab_size,
    d_model=256,
    max_seq_len=MAX_SEQ_LEN
).to(device)

# 2. Load the Riemannian Quantizer (Codebook)
# This module applies the Latent Oddity metric to discretize the continuous space.
from src.model.latent_oddity import LatentOddityQuantizer
quantizer = LatentOddityQuantizer(
    vae_model=vae_model,
    num_embeddings=VQ_CODEBOOK_SIZE,
    embedding_dim=256,
    decay=0.99
).to(device)

# 3. Load trained weights from Stage 1 & 2
try:
    vae_model.load_state_dict(torch.load(PATH_CONTINUOUS_VAE, map_location=device))
    quantizer.load_state_dict(torch.load(PATH_ODDITY_QUANTIZER, map_location=device))

    # Set models to evaluation mode since we are only using them for inference/data processing
    vae_model.eval()
    quantizer.eval()
    print("✅ Successfully loaded Riemannian Latent Oddity models.")
except FileNotFoundError as e:
    print(f"❌ ERROR: Missing weights. {e}")

Using device: cuda
Loading tokenizer: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

✅ Successfully loaded Riemannian Latent Oddity models.


### 2. Create the Riemannian Assorted Dataset

We now fetch the raw mathematical dataset (`MetaMathQA`). The function `create_assorted_dataset_oddity` takes the raw text, encodes it through the VAE, and discretizes the representations using the Riemannian metric.

The resulting output is a `.jsonl` file containing sequences of discrete tokens that strictly follow the true geometrical shape of the data manifold.

In [ ]:
from datasets import load_dataset
from src.utils import create_assorted_dataset_oddity

# 1. Load MetaMathQA dataset
# Use a random subset of 10,000 samples for processing
raw_dataset = load_dataset("meta-math/MetaMathQA")['train'].shuffle(seed=42).select(range(10000))

# 2. Adaptation: MetaMathQA uses 'query' and 'response'
# We map these to the expected internal 'question' and 'answer' format
# required by the create_assorted_dataset_oddity function.
print("Adapting MetaMathQA fields...")

# 3. Create the Riemannian Assorted Dataset
# This function maps Chain-of-Thought (CoT) text to Riemannian discrete tokens
# based on the decoder's manifold curvature.
assorted_samples = create_assorted_dataset_oddity(
    vae_model=vae_model,
    quantizer_model=quantizer,
    llm_tokenizer=tokenizer,
    dataset=raw_dataset,
    device=device
)

print(f"\nGenerated {len(assorted_samples)} Riemannian assorted samples from MetaMathQA.")

# Define the output path to save the processed dataset
PATH_PROCESSED_DATA_ODDITY = "/content/drive/MyDrive/DLAI/data/processed/metamath_assorted_oddity_uv.jsonl"

print(f"Saving MetaMath Riemannian data to {PATH_PROCESSED_DATA_ODDITY}...")
# Ensure the target directory exists before saving
os.makedirs(os.path.dirname(PATH_PROCESSED_DATA_ODDITY), exist_ok=True)

# Write the processed data into a JSON Lines file
with open(PATH_PROCESSED_DATA_ODDITY, 'w') as f:
    for item in assorted_samples:
        f.write(json.dumps(item) + '\n')

print("✅ MetaMath Riemannian Processed data saved.")

README.md: 0.00B [00:00, ?B/s]

MetaMathQA-395K.json:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

Adapting MetaMathQA fields...
🚀 Creating Riemannian 'Latent Oddity' Assorted Dataset...


  0%|          | 0/10000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
100%|██████████| 10000/10000 [04:27<00:00, 37.33it/s]



Generated 9987 Riemannian assorted samples from MetaMathQA.
Saving MetaMath Riemannian data to /content/drive/MyDrive/DLAI/data/processed/metamath_assorted_oddity_uv.jsonl...
✅ MetaMath Riemannian Processed data saved.
